# input

In [1]:
import mne
import numpy as np
from scipy.signal import welch, coherence
from itertools import combinations

# ======================
# 1️⃣ Load and preprocess
# ======================
fname_main = "/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main.fif"
fname_rest = "/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/resting.fif"

raw_main = mne.io.read_raw_fif(fname_main, preload=True)
sfreq = raw_main.info['sfreq']
#
# raw_rest = mne.io.read_raw_fif(fname_rest, preload=True)
#
# raw_main.filter(1., 40.)
# raw_rest.filter(1., 40.)
#
# # Pick magnetometers
# picks_main = mne.pick_types(raw_main.info, meg='mag')
# picks_rest = mne.pick_types(raw_rest.info, meg='mag')
#
# ch_names_main = [raw_main.ch_names[i] for i in picks_main]
# ch_names_rest = [raw_rest.info['ch_names'][i] for i in picks_rest]
#
# common_mags = list(set(ch_names_main).intersection(set(ch_names_rest)))
# assert len(common_mags) > 0, "No common mag channels found."
# print("Using channels:", common_mags)
#
# data_main = raw_main.copy().pick(common_mags).get_data()
# data_rest = raw_rest.copy().pick(common_mags).get_data()
# sfreq = raw_main.info['sfreq']

Opening raw data file /Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main.fif...
    Read a total of 8 projection items:
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
    Range : 55000 ... 1630999 =     55.000 ...  1630.999 secs
Ready.
Opening raw data file /Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main-1.fif...
    Read a total of 8 projection items:
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
   

/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_39363/2043681240.py:12: RuntimeWarning: This filename (/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_main = mne.io.read_raw_fif(fname_main, preload=True)


Reading 0 ... 2542999  =      0.000 ...  2542.999 secs...


# window gen

In [6]:
import os
import gc
import numpy as np
import mne

# ======================
# Parameters
# ======================
WINDOW_SEC = 2
TEST_FRAC = 0.2

def create_nonoverlapping_windows(data, window_samples):
    """
    Return only full windows, discarding leftover samples at the end.
    """
    n_channels, n_times = data.shape
    n_windows = n_times // window_samples
    return [data[:, i*window_samples:(i+1)*window_samples] for i in range(n_windows)]

def split_and_window(data, window_samples, test_frac=0.2):
    """
    Split by time fraction, then create full windows per split.
    """
    n_samples = data.shape[1]
    split_idx = int(n_samples * (1 - test_frac))
    train_windows = create_nonoverlapping_windows(data[:, :split_idx], window_samples)
    test_windows  = create_nonoverlapping_windows(data[:, split_idx:], window_samples)
    return train_windows, test_windows

# ======================
# Multi-subject processing
# ======================
def process_subjects_windows(fname_main, fname_rest, subject_id, out_dir="windows", window_sec=2, test_frac=0.2):
    """
    Reads one subject, splits into windows, and saves them to disk.
    """
    os.makedirs(out_dir, exist_ok=True)

    # Load raw
    raw_main = mne.io.read_raw_fif(fname_main, preload=True)
    raw_rest = mne.io.read_raw_fif(fname_rest, preload=True)
    raw_main.filter(1., 40.)
    raw_rest.filter(1., 40.)

    # Common magnetometers
    picks_main = mne.pick_types(raw_main.info, meg='mag')
    picks_rest = mne.pick_types(raw_rest.info, meg='mag')
    ch_names_main = [raw_main.ch_names[i] for i in picks_main]
    ch_names_rest = [raw_rest.info['ch_names'][i] for i in picks_rest]
    common_mags = list(set(ch_names_main).intersection(set(ch_names_rest)))
    assert len(common_mags) > 0, f"No common mag channels for subject {subject_id}"

    # Extract data arrays
    data_main = raw_main.copy().pick(common_mags).get_data()
    data_rest = raw_rest.copy().pick(common_mags).get_data()
    sfreq = raw_main.info['sfreq']

    # Free Raw objects
    del raw_main, raw_rest
    gc.collect()

    window_samples = int(window_sec * sfreq)

    # Split and window
    train_main, test_main = split_and_window(data_main, window_samples, test_frac)
    train_rest, test_rest = split_and_window(data_rest, window_samples, test_frac)

    # Save windows
    np.save(os.path.join(out_dir, f"train_thinking_{subject_id}.npy"), np.array(train_main))
    np.save(os.path.join(out_dir, f"test_thinking_{subject_id}.npy"),  np.array(test_main))
    np.save(os.path.join(out_dir, f"train_rest_{subject_id}.npy"),     np.array(train_rest))
    np.save(os.path.join(out_dir, f"test_rest_{subject_id}.npy"),      np.array(test_rest))

    # Free memory
    del data_main, data_rest, train_main, test_main, train_rest, test_rest
    gc.collect()

In [7]:
main_files = [
    "/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main.fif",
    "/Users/roman/PycharmProjects/brain_data/data/shalamkova_alisa/main.fif",
    "/Users/roman/PycharmProjects/brain_data/data/sinitsin_ivan/words.fif",
    "/Users/roman/PycharmProjects/brain_data/data/20.08.24/2/240820_gromov/main.fif",
]
rest_files = [
    "/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/resting.fif",
    "/Users/roman/PycharmProjects/brain_data/data/shalamkova_alisa/resting_state.fif",
    "/Users/roman/PycharmProjects/brain_data/data/sinitsin_ivan/resting_state.fif",
    "/Users/roman/PycharmProjects/brain_data/data/20.08.24/2/240820_gromov/resting_state.fif",
]

for i, (f_main, f_rest) in enumerate(zip(main_files, rest_files)):
    process_subjects_windows(f_main, f_rest, subject_id=i, out_dir="windows")

Opening raw data file /Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main.fif...
    Read a total of 8 projection items:
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
    Range : 55000 ... 1630999 =     55.000 ...  1630.999 secs
Ready.
Opening raw data file /Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main-1.fif...
    Read a total of 8 projection items:
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
   

/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_39363/3941255034.py:40: RuntimeWarning: This filename (/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/main.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_main = mne.io.read_raw_fif(fname_main, preload=True)


Opening raw data file /Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/resting.fif...
    Read a total of 8 projection items:
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
    Range : 69000 ... 260999 =     69.000 ...   260.999 secs
Ready.
Reading 0 ... 191999  =      0.000 ...   191.999 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_39363/3941255034.py:41: RuntimeWarning: This filename (/Users/roman/PycharmProjects/brain_data/data/15.10.24/2/chistova_alena/resting.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_rest = mne.io.read_raw_fif(fname_rest, preload=True)


Filtering raw data in 3 contiguous segments
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge:

...

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 

/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_39363/3941255034.py:40: RuntimeWarning: This filename (/Users/roman/PycharmProjects/brain_data/data/20.08.24/2/240820_gromov/main.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_main = mne.io.read_raw_fif(fname_main, preload=True)


Opening raw data file /Users/roman/PycharmProjects/brain_data/data/20.08.24/2/240820_gromov/resting_state.fif...
    Read a total of 8 projection items:
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
        generated with autossp-1.0.1 (1 x 306)  idle
    Range : 162000 ... 399999 =    162.000 ...   399.999 secs
Ready.
Reading 0 ... 237999  =      0.000 ...   237.999 secs...


/var/folders/xx/gr06b4bj2cz8c5ht9dyhgzlr0000gn/T/ipykernel_39363/3941255034.py:41: RuntimeWarning: This filename (/Users/roman/PycharmProjects/brain_data/data/20.08.24/2/240820_gromov/resting_state.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_rest = mne.io.read_raw_fif(fname_rest, preload=True)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 

In [8]:
import numpy as np
import glob

# ======================
# 1️⃣ Paths
# ======================
window_dir = "windows"  # folder where you saved per-subject windows

# ======================
# 2️⃣ Collect file lists
# ======================
train_thinking_files = sorted(glob.glob(f"{window_dir}/train_thinking_*.npy"))
test_thinking_files  = sorted(glob.glob(f"{window_dir}/test_thinking_*.npy"))
train_rest_files     = sorted(glob.glob(f"{window_dir}/train_rest_*.npy"))
test_rest_files      = sorted(glob.glob(f"{window_dir}/test_rest_*.npy"))

# ======================
# 3️⃣ Load all windows and labels
# ======================
def load_windows_and_labels(thinking_files, rest_files):
    windows_list = []
    labels_list  = []

    # Thinking
    for f in thinking_files:
        w = np.load(f, mmap_mode='r')  # memory-mapped, avoids RAM overload
        windows_list.append(w)
        labels_list.append(np.ones(len(w), dtype=np.int8))

    # Rest
    for f in rest_files:
        w = np.load(f, mmap_mode='r')
        windows_list.append(w)
        labels_list.append(np.zeros(len(w), dtype=np.int8))

    X = np.vstack(windows_list)
    y = np.concatenate(labels_list)
    return X, y

X_train, y_train = load_windows_and_labels(train_thinking_files, train_rest_files)
X_test,  y_test  = load_windows_and_labels(test_thinking_files,  test_rest_files)

print("Train windows:", X_train.shape, "Test windows:", X_test.shape)
print("Train labels:", y_train.shape, "Test labels:", y_test.shape)

Train windows: (4443, 102, 2000) Test windows: (1108, 102, 2000)
Train labels: (4443,) Test labels: (1108,)


# features

## band feat

In [9]:
from scipy.signal import welch, butter, filtfilt

bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 40)
}

def compute_psd_batch(windows, sfreq):
    freqs, psd = welch(
        windows,
        fs=sfreq,
        nperseg=int(sfreq*2),
        axis=2
    )
    return freqs, psd

def bandpower_from_psd(freqs, psd):
    total_power = np.trapezoid(psd, freqs, axis=2)
    feats = []

    for low, high in bands.values():
        idx = (freqs >= low) & (freqs <= high)
        band_power = np.trapezoid(psd[:, :, idx], freqs[idx], axis=2)
        feats.append(band_power / (total_power + 1e-10))

    return np.concatenate(feats, axis=1)

def bandpass(data, sfreq, low, high):
    b, a = butter(4, [low/(sfreq/2), high/(sfreq/2)], btype='band')
    return filtfilt(b, a, data, axis=1)

def fast_connectivity(window):
    corr = np.corrcoef(window)
    return corr[np.triu_indices_from(corr, k=1)]

def coherence_band_fast(window, sfreq, band):
    filtered = bandpass(window, sfreq, band[0], band[1])
    return fast_connectivity(filtered)

## entropy, etc

In [10]:
def spectral_entropy(psd):
    psd_norm = psd / (np.sum(psd, axis=-1, keepdims=True) + 1e-10)
    return -np.sum(psd_norm * np.log(psd_norm + 1e-10), axis=-1)

def hjorth_params(signal):
    diff1 = np.diff(signal)
    diff2 = np.diff(diff1)

    var0 = np.var(signal)
    var1 = np.var(diff1)
    var2 = np.var(diff2)

    activity = var0
    mobility = np.sqrt(var1 / (var0 + 1e-10))
    complexity = np.sqrt(var2 / (var1 + 1e-10)) / (mobility + 1e-10)

    return np.array([activity, mobility, complexity])

## coordinate feat

In [11]:
def add_coordinate_features(raw_info, window):
    mag_picks = mne.pick_types(raw_info, meg='mag')
    positions = np.array([raw_info['chs'][i]['loc'][:3] for i in mag_picks])

    # Activity per channel
    activity = np.mean(window**2, axis=1)
    activity /= (np.sum(activity) + 1e-10)

    center = np.sum(positions * activity[:, None], axis=0)
    spread = np.sum(((positions - center)**2) * activity[:, None], axis=0)

    return np.concatenate([center, spread])

## features pipeline run

In [12]:
# ======================
# Fast feature extraction for all bands
# ======================
def extract_features_fast(windows, raw_info, sfreq):
    """
    windows: array (n_windows, n_channels, n_times)
    raw_info: MNE Raw.info
    sfreq: sampling frequency
    """
    # --- Precompute PSD for all windows ---
    freqs, psd = compute_psd_batch(windows, sfreq)  # shape: (n_windows, n_channels, n_freqs)

    # --- Bandpower features ---
    bp = bandpower_from_psd(freqs, psd)  # returns (n_windows, n_channels * n_bands)

    # --- Spectral entropy per channel ---
    ent = spectral_entropy(psd)  # (n_windows, n_channels)

    # --- Connectivity for all bands ---
    bands_list = {
        "delta": (1,4),
        "theta": (4,8),
        "alpha": (8,13),
        "beta":  (13,30),
        "gamma": (30,40)
    }

    conn_features = []
    for band_name, band_range in bands_list.items():
        # Compute coherence for this band for each window
        conn_band = np.array([coherence_band_fast(w, sfreq, band_range) for w in windows])
        conn_features.append(conn_band)
    conn_all = np.concatenate(conn_features, axis=1)  # shape: (n_windows, n_conn * n_bands)

    # --- Hjorth parameters per channel ---
    hjorth_feats = np.array([
        np.concatenate([hjorth_params(ch) for ch in w])
        for w in windows
    ])  # shape: (n_windows, n_channels*3)

    # --- Coordinate / spatial features ---
    coord_feats = np.array([add_coordinate_features(raw_info, w) for w in windows])  # (n_windows, n_coord)

    # --- Combine all ---
    X = np.concatenate([bp, ent, conn_all, hjorth_feats, coord_feats], axis=1)
    return X

X_train_feat = extract_features_fast(X_train, raw_main.info, sfreq)
X_test_feat  = extract_features_fast(X_test, raw_main.info, sfreq)

print(X_train_feat.shape)

/Users/roman/PycharmProjects/brain_data/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


(4443, 26679)


## scaler, prep

In [13]:
from sklearn.preprocessing import StandardScaler
from collections import Counter

scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

counts = Counter(y_train)
scale_pos_weight = counts[0] / counts[1]

print("Class ratio:", counts)

Class ratio: Counter({np.int8(1): 4147, np.int8(0): 296})


# model - XGBClassifier - train

In [14]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    # scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    n_jobs=-1
)

model.fit(X_train_feat, y_train)

preds = model.predict(X_test_feat)
probs = model.predict_proba(X_test_feat)[:,1]

print(classification_report(y_test, preds))
print("ROC AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.82      0.73      0.77        73
           1       0.98      0.99      0.98      1035

    accuracy                           0.97      1108
   macro avg       0.90      0.86      0.88      1108
weighted avg       0.97      0.97      0.97      1108

ROC AUC: 0.982026338428959


this is with all bands in coherency, as well as no target leaking in train-test split
```
              precision    recall  f1-score   support

         0.0       0.89      0.89      0.89        19
         1.0       0.99      0.99      0.99       254

    accuracy                           0.99       273
   macro avg       0.94      0.94      0.94       273
weighted avg       0.99      0.99      0.99       273

ROC AUC: 0.9970990468296725
```

# feat importances

In [49]:
import pandas as pd

feature_importance = model.feature_importances_

n_channels = len(common_mags)
n_bands = len(bands)

bp_size = n_channels * n_bands

bp_importance = feature_importance[:bp_size].reshape(n_bands, n_channels)

df = pd.DataFrame(
    bp_importance,
    index=list(bands.keys()),
    columns=common_mags
)

print(df.T.sort_values(by="alpha", ascending=False).head(15))

         delta     theta     alpha  beta     gamma
MEG2521    0.0  0.000492  0.000714   0.0  0.000000
MEG2431    0.0  0.000000  0.000394   0.0  0.000000
MEG1031    0.0  0.000000  0.000000   0.0  0.000000
MEG1941    0.0  0.000000  0.000000   0.0  0.000000
MEG2021    0.0  0.000000  0.000000   0.0  0.000000
MEG2211    0.0  0.000000  0.000000   0.0  0.000000
MEG2641    0.0  0.000000  0.000000   0.0  0.000000
MEG2041    0.0  0.000000  0.000000   0.0  0.000000
MEG2121    0.0  0.000000  0.000000   0.0  0.000862
MEG1541    0.0  0.000000  0.000000   0.0  0.000000
MEG0931    0.0  0.000000  0.000000   0.0  0.000000
MEG0521    0.0  0.000000  0.000000   0.0  0.000000
MEG0641    0.0  0.000000  0.000000   0.0  0.000000
MEG1041    0.0  0.000000  0.000000   0.0  0.000000
MEG0541    0.0  0.000000  0.000000   0.0  0.000000


In [50]:
for band in bands.keys():
    print('-'*50)
    print(band)
    print(df.T[df.T[band] > 0])

--------------------------------------------------
delta
            delta  theta  alpha  beta     gamma
MEG1721  0.006419    0.0    0.0   0.0  0.000427
MEG1611  0.005978    0.0    0.0   0.0  0.000000
--------------------------------------------------
theta
         delta     theta     alpha  beta  gamma
MEG2521    0.0  0.000492  0.000714   0.0    0.0
MEG0511    0.0  0.000483  0.000000   0.0    0.0
--------------------------------------------------
alpha
         delta     theta     alpha  beta  gamma
MEG2431    0.0  0.000000  0.000394   0.0    0.0
MEG2521    0.0  0.000492  0.000714   0.0    0.0
--------------------------------------------------
beta
Empty DataFrame
Columns: [delta, theta, alpha, beta, gamma]
Index: []
--------------------------------------------------
gamma
            delta  theta  alpha  beta     gamma
MEG1721  0.006419    0.0    0.0   0.0  0.000427
MEG1731  0.000000    0.0    0.0   0.0  0.000617
MEG1931  0.000000    0.0    0.0   0.0  0.018091
MEG2531  0.000000    0

In [ ]:
df.T[df.T['delta'] > 0]

# positions breakdown

In [10]:
positions = np.array([
    raw_main.info['chs'][raw_main.ch_names.index(ch)]['loc'][:3]
    for ch in common_mags
])

channel_df = pd.DataFrame({
    "channel": common_mags,
    "x": positions[:,0],
    "y": positions[:,1],
    "z": positions[:,2]
})

def assign_region(y):
    if y > 0.03:
        return "frontal"
    elif y > -0.03:
        return "central"
    else:
        return "parietal"

channel_df["region"] = channel_df["y"].apply(assign_region)

channel_df.head()

,channel,x,y,z,region
0,MEG1031,0.0368,0.0752,0.0923,frontal
1,MEG1721,-0.0861,-0.0660,-0.0282,parietal
2,MEG1321,0.1074,0.0329,0.0081,frontal
3,MEG2131,0.0170,-0.1098,-0.0618,parietal
4,MEG0821,0.0001,0.1316,0.0500,frontal


# feat importances - results

In [25]:
import numpy as np
import pandas as pd

fi = model.feature_importances_  # length = X_train_feat.shape[1]

# sizes
n_channels = len(common_mags)
n_bands = len(bands)
n_conn = n_channels*(n_channels-1)//2
n_hjorth = n_channels*3
n_coord = 6

# Bandpower
bp_fi = fi[:n_channels*n_bands].reshape(n_bands, n_channels)
bands_list = list(bands.keys())
bp_df = pd.DataFrame(bp_fi, index=bands_list, columns=common_mags)

# Spectral entropy
ent_fi = fi[n_channels*n_bands : n_channels*n_bands + n_channels]
ent_df = pd.Series(ent_fi, index=common_mags, name="spectral_entropy")

# Connectivity alpha
conn_alpha_fi = fi[n_channels*n_bands + n_channels : n_channels*n_bands + n_channels + n_conn]
# Conn features are pairwise; can map to channel pairs:
from itertools import combinations
channel_pairs = list(combinations(common_mags, 2))
conn_alpha_df = pd.DataFrame({
    "channel_pair": channel_pairs,
    "conn_alpha": conn_alpha_fi
}).sort_values("conn_alpha", ascending=False)

# Connectivity theta
conn_theta_fi = fi[n_channels*n_bands + n_channels + n_conn : n_channels*n_bands + n_channels + n_conn*2]
conn_theta_df = pd.DataFrame({
    "channel_pair": channel_pairs,
    "conn_theta": conn_theta_fi
}).sort_values("conn_theta", ascending=False)

# Hjorth
hj_fi = fi[n_channels*n_bands + n_channels + n_conn*2 : n_channels*n_bands + n_channels + n_conn*2 + n_hjorth]
hj_df = pd.DataFrame(hj_fi.reshape(n_channels, 3), index=common_mags, columns=["activity", "mobility", "complexity"])

# Coordinate features
coord_fi = fi[-n_coord:]
coord_names = ["center_x","center_y","center_z","spread_x","spread_y","spread_z"]
coord_df = pd.Series(coord_fi, index=coord_names, name="coord")

## spectral entropy

In [29]:
ent_df[ent_df > 0]

Series([], Name: spectral_entropy, dtype: float32)

## connectivity (theta and alpha)

In [32]:
conn_alpha_df

,channel_pair,conn_alpha
1740,"(MEG2511, MEG1921)",0.051243
4345,"(MEG0341, MEG0541)",0.008983
1698,"(MEG2511, MEG2141)",0.007779
4548,"(MEG2521, MEG1921)",0.007394
10,"(MEG1031, MEG0121)",0.007218
...,...,...
1742,"(MEG2511, MEG1011)",0.000000
1741,"(MEG2511, MEG0431)",0.000000
1739,"(MEG2511, MEG1641)",0.000000
1738,"(MEG2511, MEG0511)",0.000000


In [33]:
conn_alpha_df[conn_alpha_df['conn_alpha'] > 0]

,channel_pair,conn_alpha
1740,"(MEG2511, MEG1921)",0.051243
4345,"(MEG0341, MEG0541)",0.008983
1698,"(MEG2511, MEG2141)",0.007779
4548,"(MEG2521, MEG1921)",0.007394
10,"(MEG1031, MEG0121)",0.007218
...,...,...
1590,"(MEG0721, MEG1931)",0.000033
3093,"(MEG2011, MEG2431)",0.000027
683,"(MEG0211, MEG2411)",0.000018
538,"(MEG2221, MEG0231)",0.000016


Occipital - MEG2511, 1921 too

In [35]:
conn_theta_df[conn_theta_df['conn_theta'] > 0]

,channel_pair,conn_theta
646,"(MEG0211, MEG1411)",0.005810
816,"(MEG0421, MEG1711)",0.003756
925,"(MEG0141, MEG1411)",0.003342
3403,"(MEG0811, MEG0641)",0.003217
349,"(MEG2131, MEG2031)",0.002767
...,...,...
1803,"(MEG0411, MEG0331)",0.000046
2898,"(MEG2111, MEG2431)",0.000030
3427,"(MEG0811, MEG1441)",0.000014
4836,"(MEG0541, MEG2611)",0.000013


frontal - 646
816 - occi + front
925 - frontal + temp
3403 - both front
349 - both occipital

## hjorn complexity parameters

In [37]:
hj_df[hj_df['activity'] > 0]

,activity,mobility,complexity
MEG1321,0.003089,0.0,0.000000
MEG0311,0.000067,0.0,0.003558
MEG2011,0.009079,0.0,0.016998
MEG1611,0.001917,0.0,0.000000
MEG2031,0.006543,0.0,0.000000
MEG1511,0.003368,0.0,0.000000
MEG1821,0.017723,0.0,0.000000
MEG1911,0.014221,0.0,0.000000
MEG1831,0.001777,0.0,0.000000


In [38]:
hj_df[hj_df['mobility'] > 0]

,activity,mobility,complexity
MEG0421,0.0,0.010867,0.013261
MEG2511,0.0,0.000524,0.000513
MEG0411,0.0,0.019952,0.000000
MEG2621,0.0,0.000005,0.000000
MEG2531,0.0,0.012594,0.047348
MEG0321,0.0,0.015855,0.000000
MEG2541,0.0,0.000928,0.000000
MEG2631,0.0,0.000357,0.000399
MEG2431,0.0,0.008111,0.000000
MEG2521,0.0,0.000908,0.035941


In [39]:
hj_df[hj_df['complexity'] > 0]

,activity,mobility,complexity
MEG1721,0.000000,0.000000,0.003975
MEG2131,0.000000,0.000000,0.001071
MEG0421,0.000000,0.010867,0.013261
MEG2511,0.000000,0.000524,0.000513
MEG1531,0.000000,0.000000,0.000245
MEG1731,0.000000,0.000000,0.000946
MEG1931,0.000000,0.000000,0.019655
MEG2531,0.000000,0.012594,0.047348
MEG1221,0.000000,0.000000,0.000410
MEG0311,0.000067,0.000000,0.003558


## spatial features

In [40]:
coord_df

center_x    0.000000
center_y    0.000000
center_z    0.014669
spread_x    0.006952
spread_y    0.010960
spread_z    0.002170
Name: coord, dtype: float32